## This is the code to train the model and acquire influence for Number of Samples Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [3]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [4]:
import random
from keras.optimizers import SGD

In [5]:
from sklearn.datasets import make_classification

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to test on different number of samples.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [6]:
train_pool = 16000
test_size = 500
n_features=10
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]
sep = 1.5

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [7]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [8]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      -2.042923   2.447225   1.459419  -4.895205  -1.931741   2.439100   
1      -3.021719  -2.189454   1.594643  -0.936749   6.580163  -0.856511   
2       1.744477  -1.747769  -1.803517  -3.028068  -0.517037   3.262006   
3      -0.372644   3.341254   1.849793   0.905480   0.359212   1.766193   
4      -1.972171  -3.053477   1.529830   1.741359   0.858712   1.675452   
...          ...        ...        ...        ...        ...        ...   
16495  -1.318844  -1.342788   0.624967  -0.463707  -0.735329   1.971186   
16496  -3.063709  -0.202935  -0.534888  -6.109804  -0.977436   1.353489   
16497   0.408280  -2.252040   2.995770  -0.442872  -0.904981   2.307891   
16498  -0.117513   3.516412  -3.900624  -2.741435  -4.439399  -0.522876   
16499   3.901795  -1.910619  -0.779033  -0.536239  -1.471359   0.685229   

       feature_7  feature_8  feature_9  feature_10  label     id  
0      -0.694749  -3.070256  -3.

In [9]:
exact_size = 8500

In [10]:
cur_ratio = ratios[4]
print(cur_ratio)

(5, 5)


In [11]:
major, minor = cur_ratio

In [12]:
df0 = df[df.label == 0]  
df1 = df[df.label == 1] 

In [13]:
# t0 = int(exact_size * major / (major + minor))
# t1 = exact_size - t0 
# print(t0,t1)

In [14]:
t0 = 4250
t1 = 4250

In [15]:
s0 = df0.sample(n=t0, random_state=seed)
s1 = df1.sample(n=t1, random_state=seed)

In [16]:
df = pd.concat([s0, s1], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [17]:
print(df)
print(df["label"].value_counts())

      feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0     -0.318733   5.396442   1.414061   3.356327   2.364601   2.342864   
1      3.344966   2.424859   5.095402   1.836602  -0.599445   0.611780   
2      2.667142  -2.090442  -1.641394  -2.687489   0.013207   5.190774   
3      1.916675  -2.365890  -1.452175  -1.875552  -0.113632   3.494301   
4     -0.025394   4.375837   1.255440   1.625592  -1.183051   4.534315   
...         ...        ...        ...        ...        ...        ...   
8495  -2.313881   1.600642   0.915895   1.690314  -1.885480   0.811767   
8496  -0.220535   0.361194  -1.581270  -1.457011   3.655984   2.179526   
8497  -1.744727   0.652012  -0.121031   1.655096   2.296067   0.245165   
8498  -1.806808  -0.704839   1.698431  -3.602001   0.844647   1.772143   
8499   3.773403  -1.464346  -2.196749  -0.057284   0.037428   1.036493   

      feature_7  feature_8  feature_9  feature_10  label     id  
0      0.207598   0.973267   1.528003   -0.05

In [18]:
# id_label_df = df[["id", "label"]].copy()
# print(id_label_df)
# id_label_df.to_csv("9_1_class_labelIDs.csv",index = False)

In [19]:
n0 = test_size //2
n1 = test_size - n0

In [20]:
g = df.groupby("label", group_keys=False)
test_df = pd.concat([
    g.get_group(0).sample(n=n0, random_state=42, replace=False),
    g.get_group(1).sample(n=n1, random_state=42, replace=False),
]).sample(frac=1, random_state=42)

train_df = df.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [21]:
print(test_df.groupby("label").get_group(0))
print(test_df["label"].value_counts())

     feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
1     2.883047   1.002061  -0.671295  -5.149648  -3.795041   0.048639   
3     1.865875  -1.226204   0.101023  -1.995297  -0.760934  -3.894209   
4    -4.512412  -1.478154  -1.219686  -3.806119   1.581033  -2.109656   
7     0.646375   0.467035  -0.615644  -2.121739  -2.427274  -4.043968   
8     1.272779  -2.190444   0.149005  -2.783320  -0.592663  -3.440075   
..         ...        ...        ...        ...        ...        ...   
492   2.377289   1.448323  -0.412233  -3.262025  -2.054614  -1.914366   
493  -2.648850  -0.062802  -0.230846  -2.721853  -4.648615   4.080240   
494   0.032059   1.959569  -2.006054   0.155124  -1.674238  -1.129145   
495   0.941921   2.889598  -0.960276  -4.110476  -3.789246  -2.958974   
499   0.252194  -1.028174  -3.222762   1.008543  -2.049209   0.628510   

     feature_7  feature_8  feature_9  feature_10  label     id  
1     0.279419   2.789051   1.948570    3.133254      0  1

In [22]:
print(train_df["label"].value_counts())

label
1    4000
0    4000
Name: count, dtype: int64


4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

**The most important thing in this experiment is the following code**:  
Based on the train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000] defined above, we could map the number of training samples with the following code. By choosing the number in [], we could modify the train set size. Therefore, only changing the following code block is enough to produce the experiment result successfully.

In [23]:
train_df["clean_label"] = train_df["label"].copy()

# Randomly select 20% of the training samples
noise_ratio = 0.20
noise_seed = 42

rng = np.random.default_rng(noise_seed)

n_noisy = int(len(train_df) * noise_ratio)

noisy_positions = rng.choice(
    len(train_df),
    size=n_noisy,
    replace=False
)

# Indicator showing which samples were corrupted
train_df["is_noisy"] = 0
train_df.loc[train_df.index[noisy_positions], "is_noisy"] = 1

# Flip the binary labels: 0 -> 1 and 1 -> 0
train_df.loc[
    train_df.index[noisy_positions],
    "label"
] = 1 - train_df.loc[
    train_df.index[noisy_positions],
    "label"
]

# Save a clearer name for the labels used during training
train_df["noisy_label"] = train_df["label"]

print("Number of training samples:", len(train_df))
print("Number of flipped labels:", train_df["is_noisy"].sum())
print("Noise ratio:", train_df["is_noisy"].mean())

print(
    train_df[
        ["id", "clean_label", "noisy_label", "is_noisy"]
    ].head()
)

Number of training samples: 8000
Number of flipped labels: 1600
Noise ratio: 0.2
      id  clean_label  noisy_label  is_noisy
0  12161            1            1         0
1   5633            1            1         0
2  10987            1            1         0
3   4264            1            1         0
4   8390            1            1         0


In [24]:
# Feature columns used by the model
selected_features = [
    col for col in train_df.columns
    if col.startswith("feature_")
]

# Preserve IDs separately
train_ids_original = train_df["id"].to_numpy()

# Scale IDs only for the influence pipeline
IDs = (
    train_ids_original
    .reshape(-1, 1)
    .astype(np.float32)
    / 1e10
)

# Model features
X_train_features = train_df[
    selected_features
].to_numpy(dtype=np.float32)

# Append the ID column, as required by your existing influence code
X_train = np.hstack([
    X_train_features,
    IDs
])

# Use the corrupted labels for training
y_train_1d = train_df[
    "noisy_label"
].to_numpy(dtype=np.int64)

y_train = to_categorical(
    y_train_1d,
    num_classes=2
)

print(X_train.shape)
print(y_train.shape)

(8000, 11)
(8000, 2)


In [25]:
# X_train = train_df.drop(columns=["label"])
# y_train = train_df["label"]
# IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
# IDs = IDs  / 1e10

# X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
# X_train = np.hstack((X_train, IDs))
# y_train = to_categorical(y_train.values,num_classes=2)

# print(X_train)

In [26]:
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)
print(y_test.shape)

(500, 11)
(500, 2)


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [27]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [28]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS
# import seaborn as sns
# import matplotlib.pyplot as plt

In [29]:
# D = pairwise_distances(X_all) 

In [30]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [31]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [32]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [33]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [34]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
initial_model = tf.keras.models.clone_model(model)
initial_model.set_weights(model.get_weights())
model_list.append(InfluenceModel(initial_model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  checkpoint_model = tf.keras.models.clone_model(model)
  checkpoint_model.set_weights(model.get_weights())
  model_list.append(InfluenceModel(checkpoint_model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.8861 - accuracy: 0.4640 - val_loss: 0.7869 - val_accuracy: 0.4700 - 670ms/epoch - 21ms/step
32/32 - 0s - loss: 0.7312 - accuracy: 0.4964 - val_loss: 0.6627 - val_accuracy: 0.5540 - 66ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6776 - accuracy: 0.5610 - val_loss: 0.6108 - val_accuracy: 0.6680 - 61ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6558 - accuracy: 0.6085 - val_loss: 0.5822 - val_accuracy: 0.7240 - 58ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6436 - accuracy: 0.6355 - val_loss: 0.5623 - val_accuracy: 0.7580 - 74ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6351 - accuracy: 0.6529 - val_loss: 0.5470 - val_accuracy: 0.7860 - 64ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6284 - accuracy: 0.6630 - val_loss: 0.5342 - val_accuracy: 0.8040 - 63ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6229 - accuracy: 0.6747 - val_loss: 0.5230 - val_accuracy: 0.8080 - 66ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6181 - accuracy: 0.6831 - val_loss: 0.5130 - val_accuracy: 0.8120 - 64ms/epoch - 2ms/step

In [35]:
train_logits = model.predict(
    X_train,
    batch_size=256,
    verbose=0
)

# Calculate one loss value per sample using the corrupted labels
per_sample_loss_fn = CategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

training_losses = per_sample_loss_fn(
    y_train,
    train_logits
).numpy()

print(training_losses.shape)
print(pd.Series(training_losses).describe())

(8000,)
count    8000.000000
mean        0.515467
std         0.533872
min         0.041870
25%         0.191294
50%         0.272923
75%         0.534549
max         3.036478
dtype: float64


In [36]:
noise_loss_df = pd.DataFrame({
    "Train_ID": train_ids_original,
    "Clean_Label": train_df["clean_label"].to_numpy(),
    "Noisy_Label": train_df["noisy_label"].to_numpy(),
    "is_noisy": train_df["is_noisy"].to_numpy(),
    "Training_Loss": training_losses
})

print(noise_loss_df.head())
print(
    noise_loss_df.groupby("is_noisy")[
        "Training_Loss"
    ].describe()
)

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss
0     12161            1            1         0       0.237024
1      5633            1            1         0       0.148631
2     10987            1            1         0       0.311004
3      4264            1            1         0       0.249344
4      8390            1            1         0       0.197998
           count      mean       std       min       25%       50%       75%  \
is_noisy                                                                       
0         6400.0  0.284502  0.190036  0.041870  0.176238  0.236694  0.331825   
1         1600.0  1.439327  0.462300  0.098165  1.149357  1.456994  1.740153   

               max  
is_noisy            
0         2.404093  
1         3.036478  


In [37]:
noise_loss_df.to_csv(
    "Noise_GroundTruth_and_TrainingLoss.csv",
    index=False
)

In [38]:
train_df.to_csv(
    "NoisyLabel_TrainingData.csv",
    index=False
)

test_df.to_csv(
    "Clean_TestData.csv",
    index=False
)

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [39]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [40]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0        12161  0.170257
1         5633  0.211974
2        10987  0.284697
3         4264  0.156308
4         8390  0.238378
...        ...       ...
7995      9284  0.136529
7996      2511 -0.467609
7997      6327  0.395694
7998     12979  0.135739
7999     10432  0.195293

[8000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [41]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0        12161  0.011375
1         5633  0.009850
2        10987  0.033239
3         4264  0.022526
4         8390  0.013124
...        ...       ...
7995      9284  0.007369
7996      2511 -0.049873
7997      6327  0.012444
7998     12979  0.007760
7999     10432  0.021048

[8000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [42]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [43]:
TracIn_sorted.to_csv("NoisyLabel_TracIn_Scores.csv",index = False)
df_sorted.to_csv("NoisyLabel_FOIF_Scores.csv",index = False)